# Molecular ground states: what actually limits VQE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/benchmarks/molecular-ground-state.ipynb)

Reproduces the benchmark published at [zksf.org/applications/chemistry](https://zksf.org/applications/chemistry/).

The interesting result here is not that VQE works. It is **why it misses chemical accuracy**, which this notebook isolates in two runs: one with noiseless expectation values, one with 4,096 shots per term.


In [ ]:
!pip install -q qiskit qiskit-aer scipy numpy

## The molecule

H2 in the STO-3G basis at 0.735 angstrom, parity mapping with two-qubit reduction. This Hamiltonian is published, so the numbers check against the literature rather than against us. The Hamiltonian is the electronic part; the total energy adds the classical nuclear repulsion.

In [ ]:
import numpy as np, time
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

TERMS = [("II", -1.052373245772859), ("IZ", 0.39793742484318045),
         ("ZI", -0.39793742484318045), ("ZZ", -0.01128010425623538),
         ("XX", 0.18093119978423156)]
NUCLEAR = 0.7199689944489797
P = {"I": np.eye(2), "X": np.array([[0,1],[1,0]]), "Y": np.array([[0,-1j],[1j,0]]),
     "Z": np.array([[1,0],[0,-1]])}

H = sum(c * np.kron(P[a], P[b]) for (a, b), c in ((l, c) for l, c in TERMS))
exact = float(np.linalg.eigvalsh(H)[0])
print(f"exact electronic {exact:.6f} Ha")
print(f"exact TOTAL      {exact + NUCLEAR:.6f} Ha   <- the literature value")

## The ansatz

In [ ]:
def ansatz(t):
    qc = QuantumCircuit(2); qc.x(0)
    qc.ry(t[0], 0); qc.ry(t[1], 1); qc.cx(0, 1); qc.ry(t[2], 0); qc.ry(t[3], 1)
    return qc

def energy_exact(t):
    psi = Statevector.from_instruction(ansatz(t)).data
    return float(np.real(np.conj(psi) @ H @ psi))

## Run 1: noiseless expectation values

No sampling at all. This tells us what the ansatz is capable of.

In [ ]:
rng = np.random.default_rng(20260902)
best = np.inf
for r in range(3):
    x0 = np.zeros(4) if r == 0 else rng.uniform(-np.pi, np.pi, 4)
    res = minimize(energy_exact, x0, method="COBYLA", options={"maxiter": 400})
    best = min(best, energy_exact(res.x))
print(f"VQE (noiseless) {best + NUCLEAR:.6f} Ha   error {best-exact:+.6f} Ha")
print(f"chemical accuracy (0.0016 Ha): {'REACHED' if abs(best-exact) < 0.0016 else 'missed'}")

## Run 2: with shots, as hardware would

Each Pauli term is measured in its own basis. **One trap to avoid:** do not report the minimum of many shot-noisy estimates. That selects the most favourable fluctuation and can return an energy *below* the true ground state, which is variationally impossible. Our first attempt at this benchmark did exactly that. The optimiser is driven by noisy values; the reported energy is evaluated exactly at the parameters it settled on.

In [ ]:
sim, SHOTS, evals = AerSimulator(), 4096, {"n": 0}

def energy_sampled(t):
    evals["n"] += 1; total = 0.0
    for label, coeff in TERMS:
        if set(label) == {"I"}: total += coeff; continue
        qc = ansatz(t)
        for q, ch in enumerate(reversed(label)):
            if ch == "X": qc.h(q)
            elif ch == "Y": qc.sdg(q); qc.h(q)
        qc.measure_all()
        counts = sim.run(qc, shots=SHOTS).result().get_counts()
        exp = sum((-1) ** sum(1 for q, ch in enumerate(reversed(label))
                              if ch != "I" and bits[::-1][q] == "1") * n
                  for bits, n in counts.items())
        total += coeff * exp / SHOTS
    return total

t0 = time.perf_counter(); best_s = np.inf
for r in range(3):
    x0 = np.zeros(4) if r == 0 else rng.uniform(-np.pi, np.pi, 4)
    res = minimize(energy_sampled, x0, method="COBYLA", options={"maxiter": 200})
    e = energy_exact(res.x)          # scored exactly, not by the noisy value
    best_s = min(best_s, e)
print(f"VQE ({SHOTS} shots) {best_s + NUCLEAR:.6f} Ha  error {best_s-exact:+.6f} Ha "
      f"({(best_s-exact)*627.509:+.3f} kcal/mol)")
print(f"chemical accuracy: {'REACHED' if abs(best_s-exact) < 0.0016 else 'NOT reached'}")
print(f"{time.perf_counter()-t0:.0f}s, {evals['n']} circuit evaluations")
print(f"variational check: {'OK' if best_s >= exact - 1e-12 else 'VIOLATED'}")

## The finding

The ansatz reaches the exact ground state perfectly. With shots it does not. **The entire error is sampling noise inside the optimisation loop**, not a limitation of the circuit.

That is the real constraint on VQE and it is rarely the one discussed. Precision costs shots, shot count scales as roughly 1/error squared, and on a molecule with hundreds of Hamiltonian terms that multiplies. Raise `SHOTS` and watch the error fall, slowly.

## Next

- [The full benchmark page](https://zksf.org/applications/chemistry/), with the analysis and the caveats
- [How we benchmark](https://zksf.org/applications/methodology/): the rules every one of these follows
- [All applications](https://zksf.org/applications/) across six sectors
- [Certification](https://zksf.org/quantum-computing-certification/): what the accuracy statements assert
